In [0]:
# 🗓️ Day 6 — Null Handling

from pyspark.sql.functions import (
    col, when, coalesce, isnull, isnan,
    count, lit
)

data = [
    ("Ravi",    "Engineering", "Pune",       55000,  28),
    ("Priya",   "HR",           None,        42000,  32),
    ("Arjun",   None,          "Delhi",      72000,  None),
    ("Sneha",   "Finance",     "Pune",        None,  30),
    ("Rohit",   "Engineering", "Mumbai",     80000,  35),
    ("Meera",   None,          "Bangalore",  39000,  27),
    ("Karan",   "Finance",      None,        55000,  None),
    (None,      "Engineering", "Pune",       91000,  33),
    ("Nitin",   "HR",          "Mumbai",      None,  31),
    ("Anjali",  "Finance",     "Bangalore",  67000,  28),
]

cols = ["name", "dept", "city", "salary", "age"]
df = spark.createDataFrame(data, cols)
display(df)

name,dept,city,salary,age
Ravi,Engineering,Pune,55000,28
Priya,HR,null,42000,32
Arjun,null,Delhi,72000,null
Sneha,Finance,Pune,null,30
Rohit,Engineering,Mumbai,80000,35
Meera,null,Bangalore,39000,27
Karan,Finance,null,55000,null
null,Engineering,Pune,91000,33
Nitin,HR,Mumbai,null,31
Anjali,Finance,Bangalore,67000,28


# 1a. Check nulls in each column
from pyspark.sql.functions import count, when, col

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

# 1b. Filter rows where specific column is null
df.filter(col("salary").isNull()).show()
df.filter(col("dept").isNull()).show()

# 1c. Filter rows where column is NOT null
df.filter(col("name").isNotNull()).show()

# 1d. Multiple null checks
df.filter(col("salary").isNull() | col("dept").isNull()).show()

In [0]:
from pyspark.sql.functions import count, when, col

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()


+----+----+----+------+---+
|name|dept|city|salary|age|
+----+----+----+------+---+
|   1|   2|   2|     2|  2|
+----+----+----+------+---+



In [0]:
df.printSchema()

root
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- city: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- age: long (nullable = true)



In [0]:
when(col("salary").isNull(), "salary")

Column<'CASE WHEN isNull(salary) THEN salary END'>

In [0]:
# 1b. Filter rows where specific column is null
df.filter(col("salary").isNull()).show()
df.filter(col("dept").isNull()).show()

+-----+-------+------+------+---+
| name|   dept|  city|salary|age|
+-----+-------+------+------+---+
|Sneha|Finance|  Pune|  NULL| 30|
|Nitin|     HR|Mumbai|  NULL| 31|
+-----+-------+------+------+---+

+-----+----+---------+------+----+
| name|dept|     city|salary| age|
+-----+----+---------+------+----+
|Arjun|NULL|    Delhi| 72000|NULL|
|Meera|NULL|Bangalore| 39000|  27|
+-----+----+---------+------+----+



In [0]:
# 1c. Filter rows where column is NOT null
df.filter(col("name").isNotNull()).show()

+------+-----------+---------+------+----+
|  name|       dept|     city|salary| age|
+------+-----------+---------+------+----+
|  Ravi|Engineering|     Pune| 55000|  28|
| Priya|         HR|     NULL| 42000|  32|
| Arjun|       NULL|    Delhi| 72000|NULL|
| Sneha|    Finance|     Pune|  NULL|  30|
| Rohit|Engineering|   Mumbai| 80000|  35|
| Meera|       NULL|Bangalore| 39000|  27|
| Karan|    Finance|     NULL| 55000|NULL|
| Nitin|         HR|   Mumbai|  NULL|  31|
|Anjali|    Finance|Bangalore| 67000|  28|
+------+-----------+---------+------+----+



In [0]:
# 1d. Multiple null checks
df.filter(col("salary").isNull() | col("dept").isNull()).show()

+-----+-------+---------+------+----+
| name|   dept|     city|salary| age|
+-----+-------+---------+------+----+
|Arjun|   NULL|    Delhi| 72000|NULL|
|Sneha|Finance|     Pune|  NULL|  30|
|Meera|   NULL|Bangalore| 39000|  27|
|Nitin|     HR|   Mumbai|  NULL|  31|
+-----+-------+---------+------+----+



In [0]:
# 2a. Drop rows where ANY column has null
df.dropna().show()

+------+-----------+---------+------+---+
|  name|       dept|     city|salary|age|
+------+-----------+---------+------+---+
|  Ravi|Engineering|     Pune| 55000| 28|
| Rohit|Engineering|   Mumbai| 80000| 35|
|Anjali|    Finance|Bangalore| 67000| 28|
+------+-----------+---------+------+---+



In [0]:
# 2b. Drop rows where ALL columns are null
df.dropna(how="all").display()

name,dept,city,salary,age
Ravi,Engineering,Pune,55000,28
Priya,HR,null,42000,32
Arjun,null,Delhi,72000,null
Sneha,Finance,Pune,null,30
Rohit,Engineering,Mumbai,80000,35
Meera,null,Bangalore,39000,27
Karan,Finance,null,55000,null
null,Engineering,Pune,91000,33
Nitin,HR,Mumbai,null,31
Anjali,Finance,Bangalore,67000,28


In [0]:
# 2c. Drop rows where specific columns have null
df.dropna(subset=["name", "salary"]).show()


+------+-----------+---------+------+----+
|  name|       dept|     city|salary| age|
+------+-----------+---------+------+----+
|  Ravi|Engineering|     Pune| 55000|  28|
| Priya|         HR|     NULL| 42000|  32|
| Arjun|       NULL|    Delhi| 72000|NULL|
| Rohit|Engineering|   Mumbai| 80000|  35|
| Meera|       NULL|Bangalore| 39000|  27|
| Karan|    Finance|     NULL| 55000|NULL|
|Anjali|    Finance|Bangalore| 67000|  28|
+------+-----------+---------+------+----+



In [0]:
# 2d. Drop rows with less than N non-null values
df.dropna(thresh=4).show()  # keep rows with at least 4 non-null values

+------+-----------+---------+------+---+
|  name|       dept|     city|salary|age|
+------+-----------+---------+------+---+
|  Ravi|Engineering|     Pune| 55000| 28|
| Priya|         HR|     NULL| 42000| 32|
| Sneha|    Finance|     Pune|  NULL| 30|
| Rohit|Engineering|   Mumbai| 80000| 35|
| Meera|       NULL|Bangalore| 39000| 27|
|  NULL|Engineering|     Pune| 91000| 33|
| Nitin|         HR|   Mumbai|  NULL| 31|
|Anjali|    Finance|Bangalore| 67000| 28|
+------+-----------+---------+------+---+



In [0]:
# 3a. Fill all nulls with a single value
df.fillna(0).show()           # fills only numeric columns
df.fillna("Unknown").show()   # fills only string columns

+------+-----------+---------+------+---+
|  name|       dept|     city|salary|age|
+------+-----------+---------+------+---+
|  Ravi|Engineering|     Pune| 55000| 28|
| Priya|         HR|     NULL| 42000| 32|
| Arjun|       NULL|    Delhi| 72000|  0|
| Sneha|    Finance|     Pune|     0| 30|
| Rohit|Engineering|   Mumbai| 80000| 35|
| Meera|       NULL|Bangalore| 39000| 27|
| Karan|    Finance|     NULL| 55000|  0|
|  NULL|Engineering|     Pune| 91000| 33|
| Nitin|         HR|   Mumbai|     0| 31|
|Anjali|    Finance|Bangalore| 67000| 28|
+------+-----------+---------+------+---+

+-------+-----------+---------+------+----+
|   name|       dept|     city|salary| age|
+-------+-----------+---------+------+----+
|   Ravi|Engineering|     Pune| 55000|  28|
|  Priya|         HR|  Unknown| 42000|  32|
|  Arjun|    Unknown|    Delhi| 72000|NULL|
|  Sneha|    Finance|     Pune|  NULL|  30|
|  Rohit|Engineering|   Mumbai| 80000|  35|
|  Meera|    Unknown|Bangalore| 39000|  27|
|  Karan|    Fi

In [0]:
# 3b. Fill per column with different values

df.fillna({
    "name": "Unknown",
    "salary": 0,
    "age": 0,
    "dept": "Unassigned",
    "city": "Remote"
}).show()

+-------+-----------+---------+------+---+
|   name|       dept|     city|salary|age|
+-------+-----------+---------+------+---+
|   Ravi|Engineering|     Pune| 55000| 28|
|  Priya|         HR|   Remote| 42000| 32|
|  Arjun| Unassigned|    Delhi| 72000|  0|
|  Sneha|    Finance|     Pune|     0| 30|
|  Rohit|Engineering|   Mumbai| 80000| 35|
|  Meera| Unassigned|Bangalore| 39000| 27|
|  Karan|    Finance|   Remote| 55000|  0|
|Unknown|Engineering|     Pune| 91000| 33|
|  Nitin|         HR|   Mumbai|     0| 31|
| Anjali|    Finance|Bangalore| 67000| 28|
+-------+-----------+---------+------+---+



In [0]:
# 3c. Fill with column average
from pyspark.sql.functions import avg
avg_salary = df.select(avg("salary")).first()[0]
df.fillna({"salary": round(avg_salary, 0)}).show()

+------+-----------+---------+------+----+
|  name|       dept|     city|salary| age|
+------+-----------+---------+------+----+
|  Ravi|Engineering|     Pune| 55000|  28|
| Priya|         HR|     NULL| 42000|  32|
| Arjun|       NULL|    Delhi| 72000|NULL|
| Sneha|    Finance|     Pune| 62625|  30|
| Rohit|Engineering|   Mumbai| 80000|  35|
| Meera|       NULL|Bangalore| 39000|  27|
| Karan|    Finance|     NULL| 55000|NULL|
|  NULL|Engineering|     Pune| 91000|  33|
| Nitin|         HR|   Mumbai| 62625|  31|
|Anjali|    Finance|Bangalore| 67000|  28|
+------+-----------+---------+------+----+



In [0]:
from pyspark.sql.functions import coalesce, lit

# coalesce returns FIRST non-null value from a list

# 4a. Fallback chain
df = df.withColumn("dept_clean",
    coalesce(col("dept"), lit("Unassigned")))

In [0]:
# 4b. Multi-level fallback
df2 = df.withColumn("location",
    coalesce(col("city"), col("dept"), lit("Unknown")))

# 4c. Real world — primary + fallback column
# e.g. use work_email, if null use personal_email
data3 = [
    ("Ravi",  "ravi@company.com",   None),
    ("Priya",  None,                "priya@gmail.com"),
    ("Arjun",  None,                 None),
]
df3 = spark.createDataFrame(data3, ["name", "work_email", "personal_email"])

df3.withColumn("contact_email",
    coalesce(col("work_email"), col("personal_email"), lit("no-email@unknown.com"))
).show()

+-----+----------------+---------------+--------------------+
| name|      work_email| personal_email|       contact_email|
+-----+----------------+---------------+--------------------+
| Ravi|ravi@company.com|           NULL|    ravi@company.com|
|Priya|            NULL|priya@gmail.com|     priya@gmail.com|
|Arjun|            NULL|           NULL|no-email@unknown.com|
+-----+----------------+---------------+--------------------+



In [0]:
# More control than fillna

df = df.withColumn("salary_clean",
    when(col("salary").isNull(), 0)
    .otherwise(col("salary")))

df = df.withColumn("dept_clean",
    when(col("dept").isNull(), "Not Assigned")
    .when(col("dept") == "", "Empty")
    .otherwise(col("dept")))

display(df)

name,dept,city,salary,age,dept_clean,salary_clean
Ravi,Engineering,Pune,55000,28,Engineering,55000
Priya,HR,null,42000,32,HR,42000
Arjun,null,Delhi,72000,null,Not Assigned,72000
Sneha,Finance,Pune,null,30,Finance,0
Rohit,Engineering,Mumbai,80000,35,Engineering,80000
Meera,null,Bangalore,39000,27,Not Assigned,39000
Karan,Finance,null,55000,null,Finance,55000
null,Engineering,Pune,91000,33,Engineering,91000
Nitin,HR,Mumbai,null,31,HR,0
Anjali,Finance,Bangalore,67000,28,Finance,67000
